In [1]:
# imports

import pandas as pd
import numpy as np
import json
import polars as pl
from pathlib import Path
# There is also auto tokenizer for general purpose but just using BERT for simplicity
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset, DatasetDict
import torch
import glob

c:\Users\luker\Projects\dissertation\TabFact-BERT-Classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

In [ ]:
# # Extract tabfact zip if notebook is in google colab

# import zipfile
# import os

# zip_path = '/content/tabfact_data.zip'
# extract_path = '/content/'

# print("Unzipping dataset...")
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_path)
# print("Unzipping completed! Files extracted to:", extract_path)

In [ ]:
# Set data path, if notebook is in google colab use content, if not it is in the directory above

# Local repository path
DATA_PATH = "../tabfact_data"

# # Colab path
# DATA_PATH = "/content/tabfact_data"

In [2]:
# Helpers to read in the json data and convert to dataframe

def load_json_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

def convert_to_dataframe(data, is_full_cleaned=False):
    record = []
    for table_id, details in data.items():
        statements = details[0]
        enatiled_statuses = details[1]
        caption = details[-1]
        
        if is_full_cleaned:
            pos_tags = details[2]
            for statement, label, statement_pos_tags in zip(statements, enatiled_statuses, pos_tags):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "label": label,
                    "caption": caption,
                    "pos_tags": statement_pos_tags
                }) 
        else:
            for statement, label in zip(statements, enatiled_statuses):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "label": label,
                    "caption": caption
                })
    return pd.DataFrame(record)

def load_and_convert_data_to_dataframe(path, is_full_cleaned=False):
    raw_data = load_json_data(path)
    
    return convert_to_dataframe(raw_data, is_full_cleaned=is_full_cleaned)

In [ ]:
# Load in table dataframe

# df_tables = (
#     pl.read_csv(
#         "../tabfact_data/tables/*.csv",
#         has_header=False,
#         separator="\1", # Non-existent delimiter so full line/file stays intact
#         new_columns=["table_content"],
#         include_file_paths="file_name",
#     )
#     .group_by("file_name", maintain_order=True) # Group all the table lines together 
#     .agg(pl.col("table_content").str.join("\n")) # Concatenate table lines together
#     .to_pandas()
# )

csv_files = glob.glob(f"{DATA_PATH}/tables/*.csv")

df_tables = (
    pl.concat([
        pl.read_csv(
            file,
            has_header=False,
            separator="\1",
            new_columns=["table_content"],
        ).with_columns(pl.lit(Path(file).name).alias("file_name"))
        for file in csv_files
    ])
    .group_by("file_name", maintain_order=True)  # Group all the table lines together
    .agg(pl.col("table_content").str.join("\n"))  # Concatenate table lines together
    .to_pandas()
)

df_tables["file_name"] = df_tables["file_name"].apply(lambda p: Path(p).name) # Only keep the table id

df_tables.shape

(16573, 2)

In [ ]:
# Load in non table data frames

df_full_cleaned = load_and_convert_data_to_dataframe(f"{DATA_PATH}/full_cleaned.json", is_full_cleaned=True)

df_full_cleaned_refuted = df_full_cleaned[df_full_cleaned['label'] == 0]
df_full_cleaned_entailed = df_full_cleaned[df_full_cleaned['label'] == 1]

df_r1_training = load_and_convert_data_to_dataframe(f"{DATA_PATH}/r1_training_all.json")
df_r2_training = load_and_convert_data_to_dataframe(f"{DATA_PATH}/r2_training_all.json")

df_training = load_and_convert_data_to_dataframe(f"{DATA_PATH}/train_examples.json")
df_test = load_and_convert_data_to_dataframe(f"{DATA_PATH}/test_examples.json")
df_validation = load_and_convert_data_to_dataframe(f"{DATA_PATH}/val_examples.json")


In [11]:
# Returns a pandas dataframe that contains a dataframe linking the table content with a statement and entailed status
def get_table_claim_df(df, whole_table=True, table_before_claim=True):
    if table_before_claim:
        df_table_claim = pd.merge(
            df_tables[["file_name", "table_content"]],
            df[["table_id", "statement", "label"]],
            left_on="file_name",
            right_on="table_id",
            how="inner" # Join matching pairs
        )
    else:
        df_table_claim = pd.merge(
            df[["table_id", "statement", "label"]],
            df_tables[["file_name", "table_content"]],
            left_on="table_id",
            right_on="file_name",
            how="inner" # Join matching pairs
        )

    if table_before_claim:
        df_table_claim["table_claim"] = "TABLE: " + df_table_claim["table_content"] + " | CLAIM: " + df_table_claim["statement"]
    else:
        df_table_claim["table_claim"] = "CLAIM: " + df_table_claim["statement"] + " | TABLE: " + df_table_claim["table_content"]

    return df_table_claim[["table_claim", "label"]]


# Get train, validation, and test data
df_train_table_claim = get_table_claim_df(df_training)
df_val_table_claim = get_table_claim_df(df_validation)
df_test_table_claim = get_table_claim_df(df_test)

print(df_train_table_claim["table_claim"].iloc[0])

TABLE: aircraft#description#max gross weight#total disk area#max disk loading
robinson r - 22#light utility helicopter#1370 lb (635 kg)#497 ft square (46.2 m square)#2.6 lb / ft square (14 kg / m square)
bell 206b3 jetranger#turboshaft utility helicopter#3200 lb (1451 kg)#872 ft square (81.1 m square)#3.7 lb / ft square (18 kg / m square)
ch - 47d chinook#tandem rotor helicopter#50000 lb (22680 kg)#5655 ft square (526 m square)#8.8 lb / ft square (43 kg / m square)
mil mi - 26#heavy - lift helicopter#123500 lb (56000 kg)#8495 ft square (789 m square)#14.5 lb / ft square (71 kg / m square)
ch - 53e super stallion#heavy - lift helicopter#73500 lb (33300 kg)#4900 ft square (460 m square)#15 lb / ft square (72 kg / m square) | CLAIM: robinson r - 22 be the aircraft with the lowest max gross weight


In [6]:
# Define BERT model and tokenizer
bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 361.12it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

In [7]:
def tokenize_function(col):
    return bert_tokenizer(col['table_claim'], padding='max_length', truncation=True)

# Convert to dataset object for BERT training
dataset_train_table_claim = Dataset.from_pandas(df_train_table_claim)
dataset_val_table_claim = Dataset.from_pandas(df_val_table_claim)

datasets = DatasetDict({
    "train": dataset_train_table_claim,
    "val": dataset_val_table_claim
})

tokenized_datasets = datasets.map(tokenize_function, batched=True)

print(datasets["train"][:5])

Map: 100%|██████████| 12792/12792 [00:02<00:00, 5313.78 examples/s]

{'table_claim': ['TABLE: aircraft#description#max gross weight#total disk area#max disk loading\nrobinson r - 22#light utility helicopter#1370 lb (635 kg)#497 ft square (46.2 m square)#2.6 lb / ft square (14 kg / m square)\nbell 206b3 jetranger#turboshaft utility helicopter#3200 lb (1451 kg)#872 ft square (81.1 m square)#3.7 lb / ft square (18 kg / m square)\nch - 47d chinook#tandem rotor helicopter#50000 lb (22680 kg)#5655 ft square (526 m square)#8.8 lb / ft square (43 kg / m square)\nmil mi - 26#heavy - lift helicopter#123500 lb (56000 kg)#8495 ft square (789 m square)#14.5 lb / ft square (71 kg / m square)\nch - 53e super stallion#heavy - lift helicopter#73500 lb (33300 kg)#4900 ft square (460 m square)#15 lb / ft square (72 kg / m square) | CLAIM: robinson r - 22 be the aircraft with the lowest max gross weight', 'TABLE: aircraft#description#max gross weight#total disk area#max disk loading\nrobinson r - 22#light utility helicopter#1370 lb (635 kg)#497 ft square (46.2 m square)#2.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(y_true=labels, y_pred=predictions)
    return {"accuracy": acc}

In [ ]:
# Training the model with corrected metrics

if torch.cuda.is_available():
    print(f"GPU Available: {torch.cuda.get_device_name(0)}")
    training_args = TrainingArguments(
        output_dir='./results',
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        fp16=True
    )

    trainer = Trainer(
        model=bert_model,
        args=training_args,
        train_dataset=tokenized_datasets['train'],
        eval_dataset=tokenized_datasets['val'],
        compute_metrics=compute_metrics
    )

    trainer.train()
    trainer.save_model("./tabfact_bert")
else:
    print("GPU NOT active. Running on CPU. Training will not commence")

c:\Users\luker\Projects\dissertation\TabFact-BERT-Classification\.venv\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


No GPU detected - Running on CPU.


In [ ]:
# # Google colab model downloading

# from google.colab import files
# import shutil

# print("Zipping model and tokenizer folders together...")
# # Create a single zip containing the model and the tokenizer
# shutil.make_archive('/content/tabfact_bert_complete', 'zip', '/content', 'tabfact_bert')
# shutil.make_archive('/content/tabfact_tokenizer_complete', 'zip', '/content', 'tabfact_bert_tokenizer')

# print("Ready for download!")
# # Trigger downloads
# files.download('/content/tabfact_bert_complete.zip')
# files.download('/content/tabfact_tokenizer_complete.zip')